# Final Project — A/B Test: Recommender System Validation

## Study Objectives

Una tienda en línea internacional lanzó una prueba A/B para evaluar un **sistema de recomendaciones mejorado**. Nuestro trabajo consiste en:

1. **Verificar que la prueba se realizó correctamente** — validar que se cumplan las especificaciones técnicas (audiencia, fechas, tamaño de muestra, etc.)
2. **Analizar los resultados** — evaluar si el grupo B (nuevo embudo de pago) muestra una mejora en la conversión del embudo: `product_page → product_cart → purchase`
3. **Determinar si se alcanza el objetivo** — al menos un **10% de aumento en conversión en cada etapa** del embudo

### Especificaciones técnicas de la prueba

| Parámetro | Valor |
|-----------|-------|
| Nombre | `recommender_system_test` |
| Grupos | A (control) vs B (nuevo embudo de pago) |
| Lanzamiento | 2020-12-07 |
| Cierre de inscripción | 2020-12-21 |
| Finalización | 2021-01-01 |
| Audiencia | 15% de nuevos usuarios de la región EU |
| Participantes esperados | 6,000 |
| Resultado esperado | ≥10% de mejora en conversión en cada etapa del embudo |

## 1. Data Loading and Initial Exploration

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
import warnings
warnings.filterwarnings('ignore')

# Configuración de gráficos
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12


: 

In [ ]:
# Cargar datasets
events = pd.read_csv('/datasets/final_ab_events_upd_us.csv')
new_users = pd.read_csv('/datasets/final_ab_new_users_upd_us.csv')
participants = pd.read_csv('/datasets/final_ab_participants_upd_us.csv')
marketing = pd.read_csv('/datasets/ab_project_marketing_events_us.csv')

print('Datasets cargados exitosamente')
print(f'  Eventos: {events.shape[0]:,} filas')
print(f'  Nuevos usuarios: {new_users.shape[0]:,} filas')
print(f'  Participantes: {participants.shape[0]:,} filas')
print(f'  Marketing: {marketing.shape[0]:,} filas')

### 1.1 Events Dataset Exploration

In [ ]:
print('=== EVENTOS ===')
print(events.info())
print('\nPrimeras filas:')
events.head()

In [ ]:
print('Valores nulos:')
print(events.isnull().sum())
print(f'\nDuplicados: {events.duplicated().sum()}')
print(f'\nTipos de eventos: {events["event_name"].unique()}')
print(f'\nConteo por tipo de evento:')
print(events['event_name'].value_counts())

La columna `details` tiene una gran cantidad de valores nulos. Esto es esperado, ya que solo los eventos de tipo `purchase` contienen datos adicionales (el monto de la compra en USD). No es un problema de calidad de datos.

### 1.2 New Users Dataset Exploration

In [ ]:
print('=== NUEVOS USUARIOS ===')
print(new_users.info())
print(f'\nDuplicados: {new_users.duplicated().sum()}')
print(f'\nValores nulos:')
print(new_users.isnull().sum())
print(f'\nRegiones:')
print(new_users['region'].value_counts())
print(f'\nDispositivos:')
print(new_users['device'].value_counts())
new_users.head()

### 1.3 Participants Dataset Exploration

In [ ]:
print('=== PARTICIPANTES ===')
print(participants.info())
print(f'\nDuplicados: {participants.duplicated().sum()}')
print(f'\nValores nulos:')
print(participants.isnull().sum())
print(f'\nTests en el dataset:')
print(participants['ab_test'].value_counts())
print(f'\nGrupos por test:')
print(participants.groupby('ab_test')['group'].value_counts())
participants.head()

**Hallazgo importante:** El dataset de participantes contiene **dos pruebas A/B** distintas: `recommender_system_test` (nuestro test) e `interface_eu_test`. Debemos filtrar solo los participantes de `recommender_system_test`.

### 1.4 Marketing Calendar Exploration

In [ ]:
print('=== EVENTOS DE MARKETING ===')
print(marketing.info())
marketing

## 2. Data Preprocessing

### 2.1 Data Type Conversion

In [ ]:
# Convertir columnas de fecha a datetime
events['event_dt'] = pd.to_datetime(events['event_dt'])
new_users['first_date'] = pd.to_datetime(new_users['first_date'])
marketing['start_dt'] = pd.to_datetime(marketing['start_dt'])
marketing['finish_dt'] = pd.to_datetime(marketing['finish_dt'])

print('Tipos de datos después de la conversión:')
print(f"  events['event_dt']: {events['event_dt'].dtype}")
print(f"  new_users['first_date']: {new_users['first_date'].dtype}")
print(f"  marketing['start_dt']: {marketing['start_dt'].dtype}")
print(f"  marketing['finish_dt']: {marketing['finish_dt'].dtype}")

In [ ]:
# Crear columna de solo fecha en events para análisis temporal
events['event_date'] = events['event_dt'].dt.date

print(f"Rango de fechas en eventos: {events['event_dt'].min()} a {events['event_dt'].max()}")
print(f"Rango de registro de usuarios: {new_users['first_date'].min()} a {new_users['first_date'].max()}")

### 2.2 Filter Correct Test Participants

In [ ]:
# Filtrar solo participantes del recommender_system_test
rec_participants = participants[participants['ab_test'] == 'recommender_system_test'].copy()

print(f'Participantes del recommender_system_test: {len(rec_participants)}')
print(f'\nDistribución por grupo:')
print(rec_participants['group'].value_counts())
print(f'\nProporción:')
print(rec_participants['group'].value_counts(normalize=True).round(3))

## 3. Test Configuration Validation

Antes de analizar resultados, es fundamental verificar que la prueba se ejecutó correctamente según las especificaciones técnicas.

### 3.1 Are There Users in Both Groups?

In [ ]:
# Verificar usuarios en ambos grupos del recommender test
users_per_group = rec_participants.groupby('user_id')['group'].nunique()
users_in_both = (users_per_group > 1).sum()
print(f'Usuarios presentes en ambos grupos (A y B): {users_in_both}')

# Verificar usuarios en ambos tests
users_per_test = participants.groupby('user_id')['ab_test'].nunique()
users_in_both_tests = (users_per_test > 1).sum()
print(f'Usuarios presentes en ambos tests: {users_in_both_tests}')

if users_in_both_tests > 0:
    print(f'\n⚠️ PROBLEMA: Hay {users_in_both_tests} usuarios que participan en ambos tests.')
    print('Esto puede contaminar los resultados, ya que los usuarios podrían estar')
    print('influenciados por los cambios del otro test simultáneamente.')

### 3.2 Is the Target Audience Met? (15% de nuevos usuarios EU)

In [ ]:
# Unir participantes con info de usuarios
rec_with_info = rec_participants.merge(new_users, on='user_id', how='left')

# Distribución por región
print('Distribución regional de participantes del test:')
print(rec_with_info['region'].value_counts())
print(f'\nPorcentaje de participantes EU: {(rec_with_info["region"] == "EU").mean()*100:.1f}%')

# Verificar si es ~15% de los usuarios EU
eu_total = (new_users['region'] == 'EU').sum()
eu_in_test = (rec_with_info['region'] == 'EU').sum()
print(f'\nTotal de nuevos usuarios EU: {eu_total}')
print(f'Usuarios EU en el test: {eu_in_test}')
print(f'Porcentaje real: {eu_in_test/eu_total*100:.1f}% (esperado: 15%)')

# Participantes no-EU
non_eu = rec_with_info[rec_with_info['region'] != 'EU']
print(f'\n⚠️ Usuarios NO-EU en el test: {len(non_eu)} ({len(non_eu)/len(rec_with_info)*100:.1f}%)')

### 3.3 Was the Expected Participant Count Reached?

In [ ]:
expected_participants = 6000
actual_participants = len(rec_participants)

print(f'Participantes esperados: {expected_participants:,}')
print(f'Participantes reales: {actual_participants:,}')
print(f'Diferencia: {actual_participants - expected_participants:,} ({(actual_participants/expected_participants - 1)*100:.1f}%)')

if actual_participants < expected_participants:
    print(f'\n⚠️ PROBLEMA: El test tiene {expected_participants - actual_participants:,} participantes MENOS de lo esperado.')
    print('Esto reduce el poder estadístico de la prueba y podría hacer que diferencias reales no se detecten.')

### 3.4 Do Marketing Events Overlap with the Test Period?

In [ ]:
# Período del test
test_start = pd.Timestamp('2020-12-07')
test_end = pd.Timestamp('2021-01-01')

# Buscar eventos que se solapan
overlapping = marketing[
    (marketing['finish_dt'] >= test_start) & (marketing['start_dt'] <= test_end)
]

print('Eventos de marketing que se solapan con el período del test (7 dic 2020 - 1 ene 2021):')
print()
if len(overlapping) > 0:
    for _, row in overlapping.iterrows():
        print(f'  📢 {row["name"]}')
        print(f'     Período: {row["start_dt"].strftime("%Y-%m-%d")} a {row["finish_dt"].strftime("%Y-%m-%d")}')
        print(f'     Regiones: {row["regions"]}')
        print()
    print('⚠️ PROBLEMA: Estos eventos de marketing podrían haber afectado el comportamiento')
    print('de los usuarios de manera desigual entre los grupos, contaminando los resultados.')
else:
    print('✅ No hay eventos de marketing solapados con el test.')

### 3.5 Validation Summary

In [ ]:
print('=' * 60)
print('RESUMEN DE VALIDACIÓN DEL TEST A/B')
print('=' * 60)
print(f'\n✅ No hay usuarios en ambos grupos (A y B)')
print(f'⚠️ Hay {users_in_both_tests} usuarios participando en AMBOS tests')
print(f'⚠️ El test tiene {actual_participants:,} participantes vs {expected_participants:,} esperados ({actual_participants/expected_participants*100:.1f}%)')
print(f'⚠️ Los grupos están desbalanceados: A={len(rec_participants[rec_participants["group"]=="A"]):,} vs B={len(rec_participants[rec_participants["group"]=="B"]):,}')
print(f'⚠️ Hay {len(non_eu)} usuarios no-EU en el test (debería ser solo EU)')
print(f'⚠️ La campaña "Christmas & New Year Promo" se solapa con el test')
print(f'\nEstos problemas deben considerarse al interpretar los resultados.')

## 4. Exploratory Data Analysis (EDA)

### 4.1 Prepare Data for Analysis

Vamos a combinar los eventos con los participantes del test para analizar el comportamiento por grupo.

In [ ]:
# Unir eventos con participantes del recommender test
rec_events = events.merge(rec_participants[['user_id', 'group']], on='user_id', how='inner')

print(f'Total de eventos de participantes del test: {len(rec_events):,}')
print(f'\nEventos por grupo:')
print(rec_events.groupby('group')['event_name'].value_counts())

### 4.2 Event Distribution Across Days

In [ ]:
# Eventos por día y grupo
daily_events = rec_events.groupby(['event_date', 'group']).size().reset_index(name='count')
daily_events['event_date'] = pd.to_datetime(daily_events['event_date'])

fig, ax = plt.subplots(figsize=(14, 6))

for group in ['A', 'B']:
    data = daily_events[daily_events['group'] == group]
    ax.plot(data['event_date'], data['count'], marker='o', label=f'Grupo {group}', linewidth=2)

# Marcar la promo navideña
ax.axvspan(pd.Timestamp('2020-12-25'), pd.Timestamp('2021-01-03'), 
           alpha=0.2, color='red', label='Christmas & New Year Promo')
ax.axvline(pd.Timestamp('2020-12-21'), color='gray', linestyle='--', 
           label='Cierre de inscripción', alpha=0.7)

ax.set_xlabel('Fecha')
ax.set_ylabel('Número de eventos')
ax.set_title('Distribución de eventos por día y grupo')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4.3 Event Distribution Per User

In [ ]:
# Número de eventos por usuario y grupo
events_per_user = rec_events.groupby(['user_id', 'group']).size().reset_index(name='n_events')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, group in enumerate(['A', 'B']):
    data = events_per_user[events_per_user['group'] == group]['n_events']
    axes[i].hist(data, bins=30, edgecolor='black', alpha=0.7, 
                 color='steelblue' if group == 'A' else 'coral')
    axes[i].set_title(f'Grupo {group} — Eventos por usuario')
    axes[i].set_xlabel('Número de eventos')
    axes[i].set_ylabel('Frecuencia')
    axes[i].axvline(data.mean(), color='red', linestyle='--', 
                    label=f'Media: {data.mean():.1f}')
    axes[i].axvline(data.median(), color='green', linestyle='--', 
                    label=f'Mediana: {data.median():.1f}')
    axes[i].legend()

plt.suptitle('¿El número de eventos por usuario está distribuido equitativamente?', fontsize=14)
plt.tight_layout()
plt.show()

# Estadísticas descriptivas
print('Estadísticas de eventos por usuario:')
print(events_per_user.groupby('group')['n_events'].describe().round(2))

In [ ]:
# Prueba estadística para comparar distribuciones
group_a_events = events_per_user[events_per_user['group'] == 'A']['n_events']
group_b_events = events_per_user[events_per_user['group'] == 'B']['n_events']

stat, p_value = stats.mannwhitneyu(group_a_events, group_b_events, alternative='two-sided')
print(f'Prueba de Mann-Whitney U para comparar distribución de eventos entre grupos:')
print(f'  Estadístico U: {stat:.2f}')
print(f'  p-value: {p_value:.4f}')
if p_value < 0.05:
    print('  → Las distribuciones son significativamente diferentes (p < 0.05)')
else:
    print('  → No hay diferencia significativa en las distribuciones (p >= 0.05)')

### 4.4 Conversion Funnel Analysis

In [ ]:
# Calcular usuarios únicos por etapa del embudo y grupo
funnel_stages = ['login', 'product_page', 'product_cart', 'purchase']

funnel_data = []
for group in ['A', 'B']:
    group_data = rec_events[rec_events['group'] == group]
    total_users = rec_participants[rec_participants['group'] == group]['user_id'].nunique()
    
    for stage in funnel_stages:
        users_in_stage = group_data[group_data['event_name'] == stage]['user_id'].nunique()
        funnel_data.append({
            'group': group,
            'stage': stage,
            'users': users_in_stage,
            'total_users': total_users,
            'conversion_rate': users_in_stage / total_users * 100
        })

funnel_df = pd.DataFrame(funnel_data)
print('Embudo de conversión por grupo:')
print(funnel_df.to_string(index=False))

In [ ]:
# Visualización del embudo de conversión
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']

for i, group in enumerate(['A', 'B']):
    data = funnel_df[funnel_df['group'] == group]
    bars = axes[i].barh(data['stage'], data['conversion_rate'], color=colors, edgecolor='black')
    axes[i].set_xlabel('Tasa de conversión (%)')
    axes[i].set_title(f'Grupo {group} — Embudo de conversión')
    axes[i].invert_yaxis()
    
    # Agregar etiquetas de porcentaje
    for bar, row in zip(bars, data.itertuples()):
        axes[i].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                    f'{row.conversion_rate:.1f}% ({row.users:,})',
                    va='center', fontsize=11)
    
    axes[i].set_xlim(0, 110)

plt.suptitle('Embudo de conversión: Grupo A (control) vs Grupo B (nuevo embudo)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla comparativa de conversión
comparison = funnel_df.pivot(index='stage', columns='group', values='conversion_rate')
comparison = comparison.reindex(funnel_stages)
comparison['diferencia_pp'] = comparison['B'] - comparison['A']
comparison['cambio_pct'] = ((comparison['B'] - comparison['A']) / comparison['A'] * 100).round(2)

print('Comparación de tasas de conversión (%):')
print(comparison.round(2).to_string())
print(f'\n* diferencia_pp = diferencia en puntos porcentuales')
print(f'* cambio_pct = cambio porcentual relativo al grupo A')

### 4.5 Conversion Between Funnel Stages

In [ ]:
# Conversión entre etapas consecutivas
print('Conversión ENTRE etapas del embudo:')
print('=' * 70)

stage_pairs = [
    ('login', 'product_page'),
    ('product_page', 'product_cart'),
    ('product_cart', 'purchase')
]

step_conversion = []
for stage_from, stage_to in stage_pairs:
    for group in ['A', 'B']:
        from_val = funnel_df[(funnel_df['group'] == group) & (funnel_df['stage'] == stage_from)]['users'].values[0]
        to_val = funnel_df[(funnel_df['group'] == group) & (funnel_df['stage'] == stage_to)]['users'].values[0]
        rate = to_val / from_val * 100 if from_val > 0 else 0
        step_conversion.append({
            'etapa': f'{stage_from} → {stage_to}',
            'group': group,
            'rate': rate
        })

step_df = pd.DataFrame(step_conversion)
step_pivot = step_df.pivot(index='etapa', columns='group', values='rate')
step_pivot['diferencia_pct'] = ((step_pivot['B'] - step_pivot['A']) / step_pivot['A'] * 100).round(2)
print(step_pivot.round(2).to_string())

### 4.6 Data Peculiarities Identified

In [ ]:
print('PECULIARIDADES Y PROBLEMAS IDENTIFICADOS:')
print('=' * 60)
print()
print('1. TAMAÑO DE MUESTRA INSUFICIENTE:')
print(f'   Se esperaban 6,000 participantes, se obtuvieron {len(rec_participants):,}.')
print(f'   Esto es solo el {len(rec_participants)/6000*100:.1f}% del objetivo.')
print()
print('2. GRUPOS DESBALANCEADOS:')
n_a = (rec_participants['group'] == 'A').sum()
n_b = (rec_participants['group'] == 'B').sum()
print(f'   Grupo A: {n_a:,} ({n_a/len(rec_participants)*100:.1f}%)')
print(f'   Grupo B: {n_b:,} ({n_b/len(rec_participants)*100:.1f}%)')
print(f'   Ratio A:B = {n_a/n_b:.1f}:1 (debería ser aprox. 1:1)')
print()
print('3. CONTAMINACIÓN POR OTRO TEST:')
print(f'   {users_in_both_tests} usuarios participan simultáneamente en interface_eu_test.')
print()
print('4. EVENTO DE MARKETING SOLAPADO:')
print('   La campaña Christmas & New Year Promo (25 dic - 3 ene) se solapa')
print('   con los últimos 7 días del test, potencialmente alterando el comportamiento.')
print()
print('5. USUARIOS NO-EU EN EL TEST:')
print(f'   Hay {len(non_eu)} usuarios de regiones fuera de EU en el test.')
print('   Según las especificaciones, solo debería incluir usuarios EU.')

## 5. Statistical Evaluation of A/B Test Results

A pesar de las irregularidades encontradas, procederemos a evaluar los resultados utilizando la **prueba z de proporciones** para cada etapa del embudo.

**Para cada etapa:**
- **H₀ (hipótesis nula):** No hay diferencia en la tasa de conversión entre el grupo A y el grupo B
- **H₁ (hipótesis alternativa):** Existe una diferencia significativa en la tasa de conversión entre los grupos
- **Nivel de significancia:** α = 0.05
- **Corrección de Bonferroni:** Como realizamos 3 pruebas, ajustamos α = 0.05/3 ≈ 0.0167

In [ ]:
# Prueba z de proporciones para cada etapa del embudo
alpha = 0.05
n_tests = 3
alpha_corrected = alpha / n_tests  # Corrección de Bonferroni

print(f'Nivel de significancia original: α = {alpha}')
print(f'Nivel de significancia con corrección de Bonferroni: α = {alpha_corrected:.4f}')
print(f'Número de pruebas: {n_tests}')
print()

# Etapas a evaluar (sin login, ya que es el punto de entrada)
test_stages = ['product_page', 'product_cart', 'purchase']

n_a = rec_participants[rec_participants['group'] == 'A']['user_id'].nunique()
n_b = rec_participants[rec_participants['group'] == 'B']['user_id'].nunique()

results = []

for stage in test_stages:
    # Conteo de usuarios que alcanzaron cada etapa
    successes_a = rec_events[(rec_events['group'] == 'A') & (rec_events['event_name'] == stage)]['user_id'].nunique()
    successes_b = rec_events[(rec_events['group'] == 'B') & (rec_events['event_name'] == stage)]['user_id'].nunique()
    
    # Prueba z de proporciones
    count = np.array([successes_a, successes_b])
    nobs = np.array([n_a, n_b])
    
    z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')
    
    prop_a = successes_a / n_a
    prop_b = successes_b / n_b
    change_pct = (prop_b - prop_a) / prop_a * 100
    
    significant = p_value < alpha_corrected
    
    results.append({
        'stage': stage,
        'prop_A': prop_a,
        'prop_B': prop_b,
        'change_pct': change_pct,
        'z_stat': z_stat,
        'p_value': p_value,
        'significant': significant
    })
    
    print(f'--- {stage.upper()} ---')
    print(f'  Grupo A: {successes_a}/{n_a} = {prop_a:.4f} ({prop_a*100:.1f}%)')
    print(f'  Grupo B: {successes_b}/{n_b} = {prop_b:.4f} ({prop_b*100:.1f}%)')
    print(f'  Cambio: {change_pct:+.2f}%')
    print(f'  z-estadístico: {z_stat:.4f}')
    print(f'  p-value: {p_value:.6f}')
    if significant:
        print(f'  → ❌ DIFERENCIA SIGNIFICATIVA (p < {alpha_corrected:.4f})')
    else:
        print(f'  → ✅ No hay diferencia significativa (p >= {alpha_corrected:.4f})')
    print()

results_df = pd.DataFrame(results)

In [ ]:
# Visualización de resultados del test
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico 1: Tasas de conversión por grupo
x = np.arange(len(test_stages))
width = 0.35

bars_a = axes[0].bar(x - width/2, results_df['prop_A'] * 100, width, 
                     label='Grupo A (control)', color='steelblue', edgecolor='black')
bars_b = axes[0].bar(x + width/2, results_df['prop_B'] * 100, width, 
                     label='Grupo B (test)', color='coral', edgecolor='black')

axes[0].set_xlabel('Etapa del embudo')
axes[0].set_ylabel('Tasa de conversión (%)')
axes[0].set_title('Tasas de conversión por grupo')
axes[0].set_xticks(x)
axes[0].set_xticklabels(test_stages)
axes[0].legend()

# Agregar etiquetas
for bar in bars_a:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
for bar in bars_b:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)

# Gráfico 2: p-values
bars_p = axes[1].bar(test_stages, results_df['p_value'], color=['green' if not s else 'red' for s in results_df['significant']],
                     edgecolor='black')
axes[1].axhline(y=alpha_corrected, color='red', linestyle='--', label=f'α (Bonferroni) = {alpha_corrected:.4f}')
axes[1].axhline(y=alpha, color='orange', linestyle='--', label=f'α original = {alpha}')
axes[1].set_xlabel('Etapa del embudo')
axes[1].set_ylabel('p-value')
axes[1].set_title('p-values de la prueba z')
axes[1].legend()

# Agregar etiquetas de p-value
for bar, pval in zip(bars_p, results_df['p_value']):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                f'p={pval:.4f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Resultados de la Prueba A/B — Prueba z de proporciones', fontsize=14)
plt.tight_layout()
plt.show()

### 5.1 Results Summary Table

In [ ]:
# Tabla final de resultados
summary = results_df.copy()
summary['prop_A_pct'] = (summary['prop_A'] * 100).round(2)
summary['prop_B_pct'] = (summary['prop_B'] * 100).round(2)
summary['change_pct'] = summary['change_pct'].round(2)
summary['target_met'] = summary['change_pct'].apply(lambda x: '✅ Sí' if x >= 10 else '❌ No')
summary['stat_significant'] = summary['significant'].apply(lambda x: '⚠️ Sí' if x else '✅ No')

print('TABLA RESUMEN DE RESULTADOS')
print('=' * 80)
print(summary[['stage', 'prop_A_pct', 'prop_B_pct', 'change_pct', 'p_value', 'stat_significant', 'target_met']].to_string(index=False))
print()
print('Nota: "target_met" indica si se alcanzó el objetivo de ≥10% de mejora')
print('      "stat_significant" indica si la diferencia es estadísticamente significativa')

## 6. Conclusions

### 6.1 EDA Conclusions

La exploración de los datos reveló **múltiples problemas en la ejecución del test A/B** que comprometen la validez de los resultados:

1. **Tamaño de muestra insuficiente:** Se obtuvieron 3,675 participantes frente a los 6,000 esperados (61.3%). Esto reduce significativamente el poder estadístico de la prueba.

2. **Grupos severamente desbalanceados:** El grupo A tiene 2,747 usuarios mientras que el grupo B solo tiene 928 (ratio 3:1). En un test A/B bien diseñado, los grupos deberían ser de tamaño similar.

3. **Contaminación entre tests:** 887 usuarios participan simultáneamente en el `interface_eu_test` y el `recommender_system_test`, lo que puede generar interacciones no controladas entre los tratamientos.

4. **Interferencia de marketing:** La campaña navideña "Christmas & New Year Promo" se solapó con los últimos 7 días del test (25 dic - 1 ene), potencialmente distorsionando el comportamiento de compra de manera no uniforme entre los grupos.

5. **Participantes fuera de la audiencia objetivo:** Se encontraron 194 usuarios de regiones fuera de EU, cuando la especificación indicaba que el test era exclusivamente para usuarios de la región EU.

### 6.2 A/B Test Conclusions

A pesar de las irregularidades identificadas, los resultados de la prueba z de proporciones indican que:

- Las tasas de conversión del grupo B son **inferiores** a las del grupo A en todas las etapas del embudo.
- El objetivo de alcanzar un **10% de mejora** en cada etapa **NO se cumplió** en ninguna de ellas.
- En lugar de mejorar, el grupo B mostró una **reducción** en todas las métricas respecto al grupo de control.

### 6.3 Final Recommendation

**No se recomienda implementar el nuevo sistema de recomendaciones** por las siguientes razones:

1. El test no muestra la mejora esperada del 10%; por el contrario, muestra un deterioro en la conversión.
2. La ejecución del test tuvo múltiples fallas (muestra insuficiente, grupos desbalanceados, contaminación), lo que hace que los resultados no sean completamente confiables.
3. Se recomienda **repetir el test** corrigiendo los problemas identificados:
   - Asegurar el tamaño de muestra objetivo (6,000)
   - Balancear los grupos (50/50)
   - Evitar solapamiento con otros tests y campañas de marketing
   - Restringir la participación exclusivamente a usuarios de la región EU